# Choosing the play-by-play columns (step 1.7a)

nflverse play-by-play has about 370 columns per play. Loading all of them for 2016 to present would waste most of Neon's 1 GB, so `raw.pbp` keeps only the columns the project needs.

This notebook profiles every column of the 2025 season, proposes a list with the reason for each, and checks the list: that the columns exist, that `(game_id, play_id)` is unique, and roughly how much space it takes.

In [1]:
import sys
from pathlib import Path

import nflreadpy as nfl
import pandas as pd

# Notebooks run with notebooks/ as the working directory; put the repo root on the path so `pipeline` imports.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from pipeline.db import nflverse_to_pandas

pbp = nflverse_to_pandas(nfl.load_pbp(seasons=[2025]))
pbp.shape

(48771, 372)

## Every column, profiled

For each column: its pandas type, the share of plays where it's empty, how many distinct values it has, and one example value.

In [2]:
profile = pd.DataFrame(
    {
        "dtype": pbp.dtypes.astype(str),
        "null_pct": (pbp.isna().mean() * 100).round(1),
        "distinct": pbp.nunique(),
        "example": pbp.apply(lambda col: col.dropna().iloc[0] if col.notna().any() else None).astype(str).str.slice(0, 40),
    }
)
print(profile.to_string())

                                        dtype  null_pct  distinct                                   example
play_id                               float64       0.0      4850                                       1.0
game_id                                   str       0.0       285                            2025_01_ARI_NO
old_game_id                               str       0.0       285                                2025090705
home_team                                 str       0.0        32                                        NO
away_team                                 str       0.0        32                                       ARI
season_type                               str       0.0         2                                       REG
week                                    Int64       0.0        22                                         1
posteam                                   str       5.6        32                                       ARI
posteam_type                

## The proposal: 60 columns

Phase 2 (step 2.8) builds one row per team per game with plays, EPA/play on offense and defense, success rate, yards/play, pass rate, turnovers, sacks, 3rd-down %, red-zone TD % and penalties, counting only real plays (no kneels or spikes). Phase 3 analysis commonly also filters out garbage time with win probability. Each group below serves those needs.

| Group | Columns | Why |
| ----- | ------- | --- |
| Keys and context | `game_id`, `play_id`, `season`, `week`, `season_type`, `posteam`, `defteam`, `posteam_type` | Primary key, joins to schedules, and which team had the ball (offense) and which defended |
| Game state | `qtr`, `down`, `ydstogo`, `yardline_100`, `goal_to_go`, `game_seconds_remaining`, `score_differential` | The situation of each play; `yardline_100` (yards from the opponent's end zone) defines the red zone (20 or less) |
| Drives | `fixed_drive`, `fixed_drive_result` | Red-zone TD % is per drive: drives that reach the red zone, and how many end in a touchdown |
| Play type and filters | `play_type`, `pass`, `rush`, `qb_dropback`, `qb_scramble`, `qb_kneel`, `qb_spike`, `aborted_play`, `two_point_attempt` | Deciding which plays count. `pass` and `rush` already exclude kneels and spikes; the flags make the filter explicit and checkable |
| Play value | `epa`, `success`, `yards_gained`, `first_down` | EPA/play, success rate, yards/play |
| Win probability | `wp`, `wpa` | Garbage-time filters (e.g. only plays with 5–95% win probability) and late-game analysis |
| Passing | `pass_attempt`, `complete_pass`, `air_yards`, `cpoe`, `pass_oe` | Pass rate, completion %, depth of target, accuracy over expected, pass rate over expected |
| Rushing | `rush_attempt` | Rush attempts as the NFL counts them |
| Turnovers and sacks | `interception`, `fumble`, `fumble_lost`, `sack` | Turnover margin (interceptions + fumbles lost) and sacks; `fumble` vs `fumble_lost` shows fumble-recovery luck |
| Scoring | `touchdown`, `pass_touchdown`, `rush_touchdown`, `return_touchdown`, `td_team`, `safety` | Touchdowns by type, and which team scored (defensive and return TDs belong to the defense) |
| Third and fourth down | `third_down_converted`, `third_down_failed`, `fourth_down_converted`, `fourth_down_failed` | 3rd-down % and 4th-down % |
| Penalties | `penalty`, `penalty_team`, `penalty_yards` | Penalties and penalty yards by team |
| Kicking | `field_goal_attempt`, `field_goal_result`, `extra_point_result`, `punt_attempt` | Field goal %, extra points, punts per game |
| Text | `desc` | The play as the NFL's gamebook describes it (a real example is printed below). Not used in any metric, but the only way to check a play by eye against a box score (step 2.8). About 4.5 MB a season |

**Left out on purpose:** player ids and names (player analytics is a later extension), the ~80 cumulative home/away EPA and WPA columns (Phase 2 sums per team itself), schedule columns repeated on every play (`roof`, `temp`, `spread_line` and so on; they live in `raw.schedules`), kick and punt return details, and `play_deleted` (0 on every 2025 play).

In [3]:
PBP_COLUMNS = {
    "keys_and_context": ["game_id", "play_id", "season", "week", "season_type", "posteam", "defteam", "posteam_type"],
    "game_state": ["qtr", "down", "ydstogo", "yardline_100", "goal_to_go", "game_seconds_remaining", "score_differential"],
    "drives": ["fixed_drive", "fixed_drive_result"],
    "play_type_and_filters": ["play_type", "pass", "rush", "qb_dropback", "qb_scramble", "qb_kneel", "qb_spike", "aborted_play", "two_point_attempt"],
    "play_value": ["epa", "success", "yards_gained", "first_down"],
    "win_probability": ["wp", "wpa"],
    "passing": ["pass_attempt", "complete_pass", "air_yards", "cpoe", "pass_oe"],
    "rushing": ["rush_attempt"],
    "turnovers_and_sacks": ["interception", "fumble", "fumble_lost", "sack"],
    "scoring": ["touchdown", "pass_touchdown", "rush_touchdown", "return_touchdown", "td_team", "safety"],
    "third_and_fourth_down": ["third_down_converted", "third_down_failed", "fourth_down_converted", "fourth_down_failed"],
    "penalties": ["penalty", "penalty_team", "penalty_yards"],
    "kicking": ["field_goal_attempt", "field_goal_result", "extra_point_result", "punt_attempt"],
    "text": ["desc"],
}
chosen = [c for group in PBP_COLUMNS.values() for c in group]

missing = [c for c in chosen if c not in pbp.columns]
duplicated = sorted({c for c in chosen if chosen.count(c) > 1})
print(f"{len(chosen)} columns chosen; missing from the data: {missing}; listed twice: {duplicated}")
print(profile.loc[chosen].to_string())

60 columns chosen; missing from the data: []; listed twice: []
                          dtype  null_pct  distinct             example
game_id                     str       0.0       285      2025_01_ARI_NO
play_id                 float64       0.0      4850                 1.0
season                    Int64       0.0         1                2025
week                      Int64       0.0        22                   1
season_type                 str       0.0         2                 REG
posteam                     str       5.6        32                 ARI
defteam                     str       5.6        32                  NO
posteam_type                str       5.6         2                away
qtr                     float64       0.0         5                 1.0
down                    float64      16.4         4                 1.0
ydstogo                 float64       0.0        37                 0.0
yardline_100            float64       7.3        99                35.0
g

## Checks on the proposal

1. `(game_id, play_id)` must be unique, since it is `raw.pbp`'s primary key.
2. The real-play filter: kneels and spikes are already outside `pass` and `rush`, and every pass or rush play has an EPA.
3. Size: the in-memory size of the chosen columns is a rough guide to the Postgres size per season.

In [4]:
print("example desc:", pbp.loc[pbp["pass"] == 1, "desc"].iloc[0])
print("duplicate keys:", pbp.duplicated(["game_id", "play_id"]).sum())
print("kneels counted as rushes:", ((pbp["qb_kneel"] == 1) & (pbp["rush"] == 1)).sum())
print("spikes counted as passes:", ((pbp["qb_spike"] == 1) & (pbp["pass"] == 1)).sum())
real = (pbp["pass"] == 1) | (pbp["rush"] == 1)
print(f"real plays: {real.sum():,} of {len(pbp):,}; with no EPA: {(real & pbp['epa'].isna()).sum()}")

mb_all = pbp.memory_usage(deep=True).sum() / 1e6
mb_chosen = pbp[chosen].memory_usage(deep=True).sum() / 1e6
mb_desc = pbp[["desc"]].memory_usage(deep=True).sum() / 1e6
seasons = 2026 - 2016 + 1
print(f"2025 in memory: all {pbp.shape[1]} columns {mb_all:.0f} MB; chosen {len(chosen)} columns {mb_chosen:.0f} MB (desc alone {mb_desc:.0f} MB)")
print(f"rough total for {seasons} seasons: {mb_chosen * seasons:.0f} MB, or {(mb_chosen - mb_desc) * seasons:.0f} MB without desc")

example desc: (14:18) (Shotgun) 1-K.Murray pass short left to 85-T.McBride pushed ob at ARI 36 for 11 yards (27-I.Yiadom).
duplicate keys: 0
kneels counted as rushes: 0
spikes counted as passes: 0
real plays: 36,155 of 48,771; with no EPA: 0
2025 in memory: all 372 columns 206 MB; chosen 60 columns 31 MB (desc alone 5 MB)
rough total for 11 seasons: 338 MB, or 284 MB without desc


## For Rob to decide

- **Add or cut anything?** Edit `PBP_COLUMNS` above (or just say which columns), and the loader in step 1.7b will use the final list.
- **Keep `desc`?** It's the largest column, but it's the only way to read a play by eye. The size check above shows its cost.

Pandas' in-memory size isn't Postgres's on-disk size; step 1.7c measures the real table with `pg_total_relation_size`.